# Python për Data Science – Sesioni 3
## Pandas: Pastrim dhe Manipulim i të Dhënave

📄 **Dataset i vetëm për gjithë sesionin:** `employees.csv`

✅ Sot do të praktikojmë:
- **Seleksion & Filtrim** – filtrim me një/disa kushte, `loc` & `iloc`
- **Pastrim i të dhënave** – missing values (NaN), `dropna`/`fillna`, dublikate, `unique`/`nunique`
- **Transformim** – ndërrim tipi me `astype`, kolona të reja, `apply()` me funksione të personalizuara
- **Agregim & Bashkim** – `groupby` + `agg()`, `merge`, `concat`, `pivot_table`


## Setup & Leximi i CSV

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/employees.csv")
df.head()


,EmployeeID,Emri,Departamenti,Qyteti,Mosha,Paga,Data_Punesimit,Rating,Bonus
0,1001,Ilir,Sales,Pejë,46,494.0,2023-01-01,4.0,29
1,1002,Bora,Ops,Tiranë,54,1785.0,2023-01-08,3.0,192
2,1003,Kren,HR,Prizren,42,1750.0,2023-01-15,3.0,198
3,1004,Ina,HR,Tiranë,40,1261.0,2023-01-22,4.0,138
4,1005,Elio,Finance,Prishtinë,39,883.0,2023-01-29,1.0,48


### Recap i shpejtë 

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   EmployeeID      64 non-null     int64  
 1   Emri            64 non-null     str    
 2   Departamenti    62 non-null     str    
 3   Qyteti          64 non-null     str    
 4   Mosha           64 non-null     int64  
 5   Paga            64 non-null     str    
 6   Data_Punesimit  64 non-null     str    
 7   Rating          63 non-null     float64
 8   Bonus           64 non-null     int64  
dtypes: float64(1), int64(3), str(5)
memory usage: 4.6 KB


In [3]:
df.describe()

,EmployeeID,Mosha,Rating,Bonus
count,64.000000,64.000000,63.000000,64.000000
mean,1029.968750,40.078125,3.492063,119.625000
std,17.688638,10.461274,0.931059,71.476381
min,1001.000000,21.000000,1.000000,14.000000
25%,1014.750000,32.000000,3.000000,59.250000
50%,1030.500000,40.000000,4.000000,110.000000
75%,1044.250000,49.250000,4.000000,181.000000
max,1060.000000,55.000000,5.000000,308.000000


## Kontroll i shpejtë i cilësisë së data-s
Në praktikë, para analizës bëjmë 3 pyetje:
1) A ka **missing values**?
2) A ka **duplicates**?
3) A janë **tipet** siç duhet?


In [4]:
df.isna().sum()

EmployeeID        0
Emri              0
Departamenti      2
Qyteti            0
Mosha             0
Paga              0
Data_Punesimit    0
Rating            1
Bonus             0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(4)

In [6]:
df.head()

,EmployeeID,Emri,Departamenti,Qyteti,Mosha,Paga,Data_Punesimit,Rating,Bonus
0,1001,Ilir,Sales,Pejë,46,494.0,2023-01-01,4.0,29
1,1002,Bora,Ops,Tiranë,54,1785.0,2023-01-08,3.0,192
2,1003,Kren,HR,Prizren,42,1750.0,2023-01-15,3.0,198
3,1004,Ina,HR,Tiranë,40,1261.0,2023-01-22,4.0,138
4,1005,Elio,Finance,Prishtinë,39,883.0,2023-01-29,1.0,48


In [7]:
df.dtypes

EmployeeID          int64
Emri                  str
Departamenti          str
Qyteti                str
Mosha               int64
Paga                  str
Data_Punesimit        str
Rating            float64
Bonus               int64
dtype: object

---
# Seleksion & Filtrim
Si zgjedhim saktësisht ato të dhëna që na duhen — me kushte të kombinuara, `loc` dhe `iloc`.

## Filtrim me një kusht
Rikujtim i shpejtë: zgjedhim vetëm rreshtat që plotësojnë **një** kusht.

In [8]:
# df[df["Paga"] > 1500].head()
# df["Paga"].median()

In [9]:
# Kujdes: Paga ka edhe tekst si "1358,00" → e konvertojmë në numerike fillimisht
df["Paga"] = pd.to_numeric(df["Paga"], errors="coerce")

# Mbushim me median NaN-të që dolën nga konvertimi
df["Paga"] = df["Paga"].fillna(df["Paga"].median())
df.isna().sum()
# # Shembull i thjeshtë: vetëm punonjësit me Paga > 1500
df[df["Paga"] > 1500].head()


,EmployeeID,Emri,Departamenti,Qyteti,Mosha,Paga,Data_Punesimit,Rating,Bonus
1,1002,Bora,Ops,Tiranë,54,1785.0,2023-01-08,3.0,192
2,1003,Kren,HR,Prizren,42,1750.0,2023-01-15,3.0,198
9,1010,Kren,Finance,Prizren,49,1880.0,2023-03-05,3.0,280
12,1013,Dren,NaN,Tiranë,46,2345.0,2023-03-26,3.0,68
15,1016,Bledi,Ops,Tiranë,36,2119.0,2023-04-16,1.0,308


## Filtrim me disa kushte (& / |)
Kombinojmë kushte me `&` (DHE) dhe `|` (OSE). **Gjithmonë** kllapa rreth çdo kushti.

**Shembull:** punonjës në **IT ose Finance**, me pagë mbi **1200**, dhe moshë midis **25–45**.

In [10]:
filtered = df[
    (df["Departamenti"].isin(["IT", "Finance"])) &
    (df["Paga"] > 1200) &
    (df["Mosha"].between(25, 45))
]
filtered[["EmployeeID","Emri","Departamenti","Mosha","Paga"]]

,EmployeeID,Emri,Departamenti,Mosha,Paga
19,1020,Lina,Finance,30,1800.0
23,1024,Klea,Finance,37,1542.0
36,1037,Luan,Finance,45,1854.0
41,1042,Sara,IT,32,1902.0
46,1047,Bora,Finance,35,1664.0
47,1048,Rina,Finance,33,1860.0
56,1057,Ana,Finance,43,2146.0
61,1042,Sara,IT,32,1902.0


### Filtrim me OSE (|) + kontroll i qytetit

In [11]:
filtered2 = df[
    ((df["Qyteti"] == "Prishtinë") | (df["Qyteti"] == "Tiranë")) &
    (df["Rating"] >= 4)
]
filtered2[["Emri","Qyteti","Rating","Paga"]].head(10)


,Emri,Qyteti,Rating,Paga
3,Ina,Tiranë,4.0,1261.0
7,Mark,Tiranë,5.0,1111.0
22,Mark,Tiranë,4.0,1392.0
29,Drin,Tiranë,4.0,1272.0
34,Dona,Prishtinë,4.0,1490.0
38,Adea,Prishtinë,4.0,554.0
46,Bora,Tiranë,4.0,1664.0
52,Dren,Tiranë,5.0,1739.0
53,Drin,Tiranë,4.0,1125.0
57,Nora,Tiranë,4.0,2341.0


## loc & iloc (zgjedhje + ndryshim vlerash)
- **`loc`** → zgjedhje me **emra** kolonash dhe etiketa rreshtash
- **`iloc`** → zgjedhje me **pozicione** (0, 1, 2, ...) — si lista Python

**Use-case real:** kemi një gabim në paga për një punonjës (ose duhet të vendosim vlerë të përkohshme).

In [12]:
# loc: zgjedh kolonat që duam për një kusht
df.loc[df["Paga"] > 2000, ["EmployeeID","Emri","Departamenti","Paga"]].head(10)


,EmployeeID,Emri,Departamenti,Paga
12,1013,Dren,NaN,2345.0
15,1016,Bledi,Ops,2119.0
16,1017,Rion,Marketing,2312.0
26,1027,Fjor,Marketing,2191.0
54,1055,Nora,IT,2001.0
56,1057,Ana,Finance,2146.0
57,1058,Nora,IT,2341.0
58,1059,Ina,Marketing,2391.0


In [13]:
# iloc: zgjedhje me pozicion (debug / inspektim i shpejtë)
# Zgjedh 5 rreshtat e parë dhe 2 kolonat e para (sipas pozicionit)
df.iloc[0:5, 0:2]


,EmployeeID,Emri
0,1001,Ilir
1,1002,Bora
2,1003,Kren
3,1004,Ina
4,1005,Elio


In [14]:
# Ndryshim vlerash me loc: vendosim një pagë të korrigjuar për një EmployeeID (shembull)
target_id = df.loc[0, "EmployeeID"]
df.loc[df["EmployeeID"] == target_id, "Paga"] = 1500

df[df["EmployeeID"] == target_id][["EmployeeID","Emri","Paga"]]


,EmployeeID,Emri,Paga
0,1001,Ilir,1500.0


---
# Pastrim i të Dhënave
Të dhëna reale vijnë me probleme: vlera që mungojnë, duplikate, formatim i papastër. I zgjidhim një nga një.

## Missing values (NaN): kontroll + dropna + fillna
Në projekte reale zakonisht:
- kontrollojmë sa mungojnë
- vendosim: **heqim** apo **zëvendësojmë**


In [15]:
df.isna().sum()

EmployeeID        0
Emri              0
Departamenti      2
Qyteti            0
Mosha             0
Paga              0
Data_Punesimit    0
Rating            1
Bonus             0
dtype: int64

### dropna() – kur mungesat janë të pakta

In [16]:
# Hiq rreshtin vetëm nëse mungon Paga OSE Departamenti.
df_drop = df.dropna(subset=["Paga", "Departamenti"])

# Po krahasojmë sa data kishim dhe sa na mbeti pas pastrimit.
df_drop.shape, df.shape


((62, 9), (64, 9))

### fillna() – zëvendësim i thjeshtë (mode / mesatare / vlerë e zgjedhur)

In [17]:
df_fill = df.copy()

# departamenti mungon -> moda (vlera më e shpeshtë)
mode_dept = df_fill["Departamenti"].mode(dropna=True)[0]
df_fill["Departamenti"] = df_fill["Departamenti"].fillna(mode_dept)

# # rating mungon -> 0 (ose një vlerë neutrale)
df_fill["Rating"] = df_fill["Rating"].fillna(0)

df_fill.isna().sum()


EmployeeID        0
Emri              0
Departamenti      0
Qyteti            0
Mosha             0
Paga              0
Data_Punesimit    0
Rating            0
Bonus             0
dtype: int64

## Dublikate: duplicated() & drop_duplicates()
**Use-case:** i njëjti rekord është importuar dy herë.


In [18]:
df_fill.duplicated().sum()


np.int64(4)

In [19]:
df_nodup = df_fill.drop_duplicates()
len(df_fill), len(df_nodup)


(64, 60)

### Dublikate sipas subset (p.sh. EmployeeID)

In [20]:
df_nodup_id = df_fill.drop_duplicates(subset=["EmployeeID"])
len(df_fill), len(df_nodup_id)


(64, 60)

## unique() & nunique() (kontroll i kategorive)
P.sh. sa departamente kemi? sa qytete?


In [21]:
df_nodup_id['Departamenti'].unique()

<StringArray>
['Sales', 'Ops', 'HR', 'Finance', 'IT', 'Marketing']
Length: 6, dtype: str

In [22]:
df_nodup_id['Departamenti'].nunique()

6

In [23]:
df_nodup_id['Qyteti'].nunique()

6

In [24]:
# BONUS: value_counts() — sa herë shfaqet secila vlerë (frekuencat)
df_nodup_id["Departamenti"].value_counts()


Departamenti
Finance      18
Marketing    10
Sales         9
HR            9
IT            9
Ops           5
Name: count, dtype: int64

---
# Transformim & Kolona të Reja
Korrigjojmë tipet, krijojmë kolona të llogaritura dhe zbatojmë funksione të personalizuara me `apply()`.

## Ndryshimi i tipit (astype / to_datetime)
**Use-case:** `Data_Punesimit` vjen si tekst, ose paga vjen si tekst.


In [25]:
df_types = df_nodup_id.copy()
df_types["Data_Punesimit"] = pd.to_datetime(df_types["Data_Punesimit"], errors="coerce")

df_types.dtypes


EmployeeID                 int64
Emri                         str
Departamenti                 str
Qyteti                       str
Mosha                      int64
Paga                     float64
Data_Punesimit    datetime64[us]
Rating                   float64
Bonus                      int64
dtype: object

## Krijimi i kolonave të reja
**Shembull:** Krijo një kolonë të re për përqindjen e Bonusit, dhe nxjerr vitin e punësimit.

In [26]:
# Nxjerrim vitin e punësimit
df_types["Viti_Punesimit"] = df_types["Data_Punesimit"].dt.year
df_types[["Data_Punesimit","Viti_Punesimit"]].head()

,Data_Punesimit,Viti_Punesimit
0,2023-01-01,2023
1,2023-01-08,2023
2,2023-01-15,2023
3,2023-01-22,2023
4,2023-01-29,2023


In [27]:
# Përqindja e bonusit ndaj pagës
df_types["Bonus_Pct"] = (df_types["Bonus"] / df_types["Paga"]) * 100
df_types["Bonus_Pct"] = df_types["Bonus_Pct"].round(0)
df_types.head()


,EmployeeID,Emri,Departamenti,Qyteti,Mosha,Paga,Data_Punesimit,Rating,Bonus,Viti_Punesimit,Bonus_Pct
0,1001,Ilir,Sales,Pejë,46,1500.0,2023-01-01,4.0,29,2023,2.0
1,1002,Bora,Ops,Tiranë,54,1785.0,2023-01-08,3.0,192,2023,11.0
2,1003,Kren,HR,Prizren,42,1750.0,2023-01-15,3.0,198,2023,11.0
3,1004,Ina,HR,Tiranë,40,1261.0,2023-01-22,4.0,138,2023,11.0
4,1005,Elio,Finance,Prishtinë,39,883.0,2023-01-29,1.0,48,2023,5.0


## apply() — funksione të personalizuara
Kur logjika është më komplekse se një llogaritje, përdorim **funksione tonat** që zbatohen rresht për rresht.

**Shembull:** klasifikojmë çdo punonjës si "Premium", "Mesatar" ose "Bazë" sipas pagës.

In [28]:
# 1. Përkufizojmë një funksion Python
def klasifiko_pagen(paga):
    if paga > 2000:
        return "Premium"
    elif paga > 1200:
        return "Mesatar"
    else:
        return "Bazë"

# 2. Aplikojmë mbi kolonën Paga
df_types["Klasa_Pages"] = df_types["Paga"].apply(klasifiko_pagen)

df_types[["Emri", "Paga", "Klasa_Pages"]].head(10)


,Emri,Paga,Klasa_Pages
0,Ilir,1500.0,Mesatar
1,Bora,1785.0,Mesatar
2,Kren,1750.0,Mesatar
3,Ina,1261.0,Mesatar
4,Elio,883.0,Bazë
5,Drin,724.0,Bazë
6,Fjor,634.0,Bazë
7,Mark,1111.0,Bazë
8,Sara,571.0,Bazë
9,Kren,1880.0,Mesatar


In [29]:
# 3. Lambda — për funksione të vogla në një rresht
# Llogarisim pagën në euro (1 EUR = 100 njësi)
df_types["Paga_EUR"] = df_types["Paga"].apply(lambda x: round(x / 100, 2))

df_types[["Emri", "Paga", "Paga_EUR"]].head()


,Emri,Paga,Paga_EUR
0,Ilir,1500.0,15.00
1,Bora,1785.0,17.85
2,Kren,1750.0,17.50
3,Ina,1261.0,12.61
4,Elio,883.0,8.83


In [30]:
# Alternativa me np.where — më e shpejtë për kushte të thjeshta (if-else të vektorizuar)
df_types["Niveli"] = np.where(df_types["Paga"] > 1500, "I lartë", "Standard")

df_types[["Emri", "Paga", "Niveli"]].head()


,Emri,Paga,Niveli
0,Ilir,1500.0,Standard
1,Bora,1785.0,I lartë
2,Kren,1750.0,I lartë
3,Ina,1261.0,Standard
4,Elio,883.0,Standard


---
# Agregim & Bashkim
Nga rreshtat individualë te **raportet përmbledhëse** me `groupby`, `merge` dhe Pivot Table.

## Raporte reale: groupby + agg()
**Use-case:** HR kërkon raport për paga sipas departamentit.


In [31]:
report_dept = df_types.groupby("Departamenti").agg(
    Nr_Punonjesish=("EmployeeID","count"),
    Paga_Min=("Paga","min"),
    Paga_Max=("Paga","max"),
    Paga_Mesatare=("Paga","mean"),
    Mosha_Mesatare=("Mosha","mean")
).sort_values("Paga_Mesatare", ascending=False)

report_dept


,Nr_Punonjesish,Paga_Min,Paga_Max,Paga_Mesatare,Mosha_Mesatare
Departamenti,,,,,
Marketing,10,745.0,2391.0,1641.200000,35.000000
HR,9,468.0,1953.0,1473.000000,38.333333
Finance,18,455.0,2345.0,1436.222222,43.111111
Ops,5,724.0,2119.0,1396.400000,37.200000
IT,9,541.0,2341.0,1243.333333,42.777778
Sales,9,554.0,1924.0,1213.666667,39.333333


### Agregim edhe me disa statistika për Rating

In [32]:
df_types.groupby("Departamenti")["Rating"].agg(["count","mean","max"]).sort_values("mean", ascending=False).reset_index()


,Departamenti,count,mean,max
0,HR,9,3.888889,5.0
1,IT,9,3.666667,5.0
2,Finance,18,3.444444,5.0
3,Marketing,10,3.400000,5.0
4,Sales,9,3.222222,4.0
5,Ops,5,2.800000,4.0


## concat() – shtojmë rreshta nga një CSV tjetër (batch i ri)
**Use-case:** na vjen një file tjetër me punonjës të rinj dhe duam t'i bashkojmë.


In [33]:
new_hire = pd.DataFrame({
    "EmployeeID": [2001],
    "Emri": ["Elira"],
    "Departamenti": ["IT"],
    "Qyteti": ["Prishtinë"],
    "Mosha": [26],
    "Paga": [1300],
    "Data_Punesimit": ["2026-01-10"],
    "Rating": [4],
    "Bonus": [120],
    "Viti_Punesimit": [2026],
    "Bonus_Pct": [9]
})

# i bëjmë kompatibile kolonat
df_concat = pd.concat([df_types, new_hire], ignore_index=True)
df_concat.tail()


,EmployeeID,Emri,Departamenti,Qyteti,Mosha,Paga,Data_Punesimit,Rating,Bonus,Viti_Punesimit,Bonus_Pct,Klasa_Pages,Paga_EUR,Niveli
56,1057,Ana,Finance,Durrës,43,2146.0,2024-01-28 00:00:00,5.0,176,2024,8.0,Premium,21.46,I lartë
57,1058,Nora,IT,Tiranë,55,2341.0,2024-02-04 00:00:00,4.0,183,2024,8.0,Premium,23.41,I lartë
58,1059,Ina,Marketing,Prishtinë,33,2391.0,2024-02-11 00:00:00,3.0,221,2024,9.0,Premium,23.91,I lartë
59,1060,Jon,HR,Prishtinë,25,1672.0,2024-02-18 00:00:00,5.0,42,2024,3.0,Mesatar,16.72,I lartë
60,2001,Elira,IT,Prishtinë,26,1300.0,2026-01-10,4.0,120,2026,9.0,NaN,NaN,NaN


## merge() – bashkimi i tabelave (use-case real)
**Use-case:** kemi një tabelë tjetër me info për departamentet (p.sh. Menaxheri dhe Budget).


In [34]:
dept_info = pd.DataFrame({
    "Departamenti": ["IT","HR","Finance","Sales","Marketing","Ops"],
    "Menaxheri": ["Ardian","Mira","Ilir","Dona","Luan","Klea"],
    "Budget_Mujor": [20000, 8000, 15000, 12000, 10000, 9000]
})

df_all = df_concat.merge(dept_info, on="Departamenti", how="left")
df_all.head()

,EmployeeID,Emri,Departamenti,Qyteti,Mosha,Paga,Data_Punesimit,Rating,Bonus,Viti_Punesimit,Bonus_Pct,Klasa_Pages,Paga_EUR,Niveli,Menaxheri,Budget_Mujor
0,1001,Ilir,Sales,Pejë,46,1500.0,2023-01-01 00:00:00,4.0,29,2023,2.0,Mesatar,15.00,Standard,Dona,12000
1,1002,Bora,Ops,Tiranë,54,1785.0,2023-01-08 00:00:00,3.0,192,2023,11.0,Mesatar,17.85,I lartë,Klea,9000
2,1003,Kren,HR,Prizren,42,1750.0,2023-01-15 00:00:00,3.0,198,2023,11.0,Mesatar,17.50,I lartë,Mira,8000
3,1004,Ina,HR,Tiranë,40,1261.0,2023-01-22 00:00:00,4.0,138,2023,11.0,Mesatar,12.61,Standard,Mira,8000
4,1005,Elio,Finance,Prishtinë,39,883.0,2023-01-29 00:00:00,1.0,48,2023,5.0,Bazë,8.83,Standard,Ilir,15000


## pivot_table (si Excel Pivot)
**Use-case:** raport i shpejtë: paga mesatare sipas departamentit dhe qytetit.


In [35]:
pivot = pd.pivot_table(
    df_all,
    values="Paga",
    index="Departamenti",
    columns="Qyteti",
    aggfunc="mean"
).round(1)

pivot


Qyteti,Durrës,Pejë,Prishtinë,Prizren,Shkup,Tiranë
Departamenti,,,,,,
Finance,1619.4,552.0,669.0,1740.7,1857.0,1385.8
HR,1049.5,1953.0,1693.5,1582.5,NaN,1326.5
IT,541.0,1210.4,1222.5,NaN,NaN,1726.0
Marketing,NaN,1789.0,1626.7,1468.0,1490.0,1739.0
Ops,1562.0,NaN,NaN,724.0,792.0,1952.0
Sales,NaN,1483.8,971.0,788.5,1469.0,NaN


---
# 🎯 Practice Time — Ushtrime për studentët

📌 **Qëllimi:** t'i përdorni funksionet vetë, si në punë reale.
Bëjini fillimisht vetë. Më poshtë janë zgjidhjet për t'i verifikuar.


---
## Ushtrim #1 — Seleksion & Filtrim

**Detyrat:**
1) Gjeni të gjithë punonjësit me **`Paga > 1200`** **DHE** **`Departamenti == "IT"`**
2) Përdorni **`loc`** për të zgjedhur vetëm kolonat `Emri` dhe `Paga` nga rezultati
3) **Sfidë:** gjeni 3 rreshtat e parë me **`iloc`** dhe krahasoni me `loc[0:3]` — cili është ndryshimi?

---
## Ushtrim #2 (Transformim) -- Krijim kolone (Seniori) me funksion

**Detyrat (përdorni `df_all`):**
1) Kontrolloni dhe (nëse ka nevojë) konvertoni kolonën `Paga` në `float`
2) Krijoni një kolonë të re **`Paga_Vjetore`** = `Paga * 12`
3) **Sfidë:** krijoni një kolonë **`Seniori`** me `apply()`:
   - `"Senior"` nëse `Viti_Punesimit <= 2018`
   - `"Mid"` nëse `Viti_Punesimit` është midis 2019 dhe 2022
   - `"Junior"` për të tjerët

---
## Ushtrimi #3(Transformim) – Krijim kolone (Bonus_flag) me np.where
*Detyrat:*
1) Krijo një kolonë **Bonus_Flag****
2) Vendos vlerën "Yes" nëse  *Bonus* >= 100, përndryshe "No"
3) Shfaq value_counts() për kolonën e re

---
## Ushtrim #4 (Transformim) – Krijim kolone (Bonus_Total) me apply(lambda)

*Detyrat:*

1) Krijoni një kolonë të re *Paga_Mujore_Neto* duke zbritur 10% nga paga:
   - Formula: Paga * 0.9

2) Krijoni një kolonë të re *Bonus_Total*:
   - Formula: Paga + Bonus

3) Shfaqni kolonat:
   - Emri
   - Paga
   - Bonus
   - Paga_Mujore_Neto
   - Bonus_Total

## Ushtrim #5 -- Agregim & Bashkim

**Detyrat:**

1) Përdorni `groupby()` për të gjetur:
   - Pagën mesatare (`mean`)
   - Pagën maksimale (`max`)
   
   për çdo `Departamenti`

2) Ruajeni rezultatin në një DataFrame të ri me emrin `df_departamentet`

3) Bashkojeni (`merge`) `df_departamentet` me DataFrame-in origjinal duke përdorur kolonën `Departamenti`

4) Shfaqni kolonat:
   - `Emri`
   - `Departamenti`
   - `Paga`
   - `mean`
   - `max`

5) **Sfidë:** Krijoni një kolonë të re `Performanca`
   - `"Mbi Mesataren"` nëse `Paga > mean`
   - `"Nen Mesataren"` përndryshe

## Ushtrim #6 -- Transformimi i të Dhënave (Pivot Table)

**Detyrat:**

1) Krijoni një `pivot table` duke përdorur `pd.pivot_table()` ku:
   - `Departamenti` të jetë si `index`
   - `Qyteti` të jetë si `columns`
   - Të llogaritet numri i punonjësve (`count`)

2) Ruajeni rezultatin në një DataFrame të ri me emrin `raporti_makro`

3) Zëvendësoni të gjitha vlerat `NaN` me `0`

4) **Sfidë:** Krijoni një pivot table të ri ku:
   - `Departamenti` të jetë si `index`
   - `Qyteti` të jetë si `columns`
   - Të llogaritet paga maksimale (`max`)
   - Rrumbullakosni vlerat me 1 shifër pas presjes dhjetore